In [4]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

df = pd.read_excel("GSAF5 (1).xls")

In [3]:
from cleaning import clean_geographic_data
df = clean_geographic_data(df)
df[["Country", "State"]].isnull().sum()

Country    0
State      0
dtype: int64

In [23]:
from cleaning import clean_year

df = clean_year(df)

df["Year"].isnull().sum()

np.int64(2)

In [28]:
from cleaning import clean_fatal

df = clean_fatal(df)

df["Fatal Y/N"].value_counts(dropna=False)

Fatal Y/N
N          4986
Y          1496
UNKNOWN     643
Name: count, dtype: Int64

In [ ]:
# ¿En qué zonas geográficas se concentran más ataques registrados?

attacks_by_location = (
    df[
        (df["Country"] != "UNKNOWN") &
        (df["State"] != "Unknown")
    ]
    .groupby(["Country", "State"])
    .size()
    .sort_values(ascending=False)
)

attacks_by_location.head(15)

Country       State                
USA           Florida                  1200
AUSTRALIA     New South Wales           524
              Queensland                357
USA           Hawaii                    348
              California                328
AUSTRALIA     Western Australia         249
SOUTH AFRICA  Kwazulu-Natal             219
              Western Cape Province     197
USA           South Carolina            176
SOUTH AFRICA  Eastern Cape Province     169
AUSTRALIA     South Australia           122
USA           North Carolina            122
AUSTRALIA     Victoria                   98
USA           Texas                      84
BRAZIL        Pernambuco                 80
dtype: int64

Los ataques registrados se concentran especialmente en determinadas regiones de Estados Unidos y Australia.

Florida (USA) destaca claramente con 1.200 ataques registrados, seguida de New South Wales (Australia) con 524. También aparecen entre las primeras posiciones Queensland (Australia) con 357, Hawaii (USA) con 348 y California (USA) con 328.

Sudáfrica también presenta regiones con un número elevado de casos registrados, como KwaZulu-Natal (219) y Western Cape Province (197).

Por tanto, observamos que los ataques registrados no se distribuyen de manera uniforme, sino que existe una fuerte concentración en determinadas regiones, especialmente de Estados Unidos, Australia y Sudáfrica.

Estos resultados muestran dónde se han registrado más ataques, no necesariamente dónde existe mayor riesgo de sufrir un ataque

In [ ]:
# Análisis de fatalidad de los ataques por país y estado

fatal_by_location = (
    df[df["Fatal Y/N"] != "UNKNOWN"]
    .groupby(["Country", "State", "Fatal Y/N"])
    .size()
    .unstack(fill_value=0)
)

fatal_by_location.head(15)

,Fatal Y/N,N,Y
Country,State,,
ADMIRALTY ISLANDS,Manus Island,1,0
AFRICA,Unknown,0,1
ALGERIA,Unknown,0,1
AMERICAN SAMOA,Tutuila Island,0,3
ANDAMAN / NICOBAR ISLANDAS,Unknown,0,1
ANDAMAN ISLANDS,Unknown,0,1
ANGOLA,West Africa,1,0
ARGENTINA,Buenos Aires Province,1,0
ARUBA,Unknown,0,1


La clasificación de los ataques por país, estado y fatalidad permite distinguir entre ataques mortales y no mortales. Sin embargo, para comparar correctamente la fatalidad entre regiones no es suficiente observar el número de casos: es necesario calcular la proporción de ataques mortales respecto al total de ataques registrados en cada zona.

Además, las regiones con muy pocos ataques deben interpretarse con precaución, ya que un único caso mortal podría producir una tasa de fatalidad del 100%

In [ ]:
# Tasa de fatalidad por país y estado

In [30]:
fatal_by_location["Total"] = (
    fatal_by_location["N"] + fatal_by_location["Y"]
)

fatal_by_location.head()

,Fatal Y/N,N,Y,Total
Country,State,,,
ADMIRALTY ISLANDS,Manus Island,1,0,1
AFRICA,Unknown,0,1,1
ALGERIA,Unknown,0,1,1
AMERICAN SAMOA,Tutuila Island,0,3,3
ANDAMAN / NICOBAR ISLANDAS,Unknown,0,1,1


In [31]:
# Cálculo de la tasa de fatalidad

fatal_by_location["Fatality Rate (%)"] = (
    fatal_by_location["Y"] / fatal_by_location["Total"] * 100
)

fatal_by_location.head()

,Fatal Y/N,N,Y,Total,Fatality Rate (%)
Country,State,,,,
ADMIRALTY ISLANDS,Manus Island,1,0,1,0.0
AFRICA,Unknown,0,1,1,100.0
ALGERIA,Unknown,0,1,1,100.0
AMERICAN SAMOA,Tutuila Island,0,3,3,100.0
ANDAMAN / NICOBAR ISLANDAS,Unknown,0,1,1,100.0


Sería poco fiable concluir que esas zonas tienen una fatalidad especialmente alta porque tenemos muy pocos casos...

In [32]:
# Tasa de fatalidad en zonas con al menos 20 ataques registrados

fatality_min_20 = fatal_by_location[
    fatal_by_location["Total"] >= 20
]

fatality_min_20 = fatality_min_20.sort_values(
    "Fatality Rate (%)",
    ascending=False
)

fatality_min_20.head(15)

Fatal Y/N                              N    Y  Total  Fatality Rate (%)
Country       State                                                    
NEW CALEDONIA South Province          11    9     20          45.000000
UNKNOWN       Unknown                 16   13     29          44.827586
BRAZIL        Pernambuco              46   30     76          39.473684
AUSTRALIA     Torres Strait           47   23     70          32.857143
SOUTH AFRICA  Kwazulu-Natal          128   49    177          27.683616
AUSTRALIA     South Australia         86   30    116          25.862069
              Queensland             244   83    327          25.382263
              New South Wales        365  104    469          22.174840
NEW ZEALAND   North Island            54   14     68          20.588235
AUSTRALIA     Tasmania                30    7     37          18.918919
SOUTH AFRICA  Western Cape Province  151   35    186          18.817204
AUSTRALIA     Western Australia      185   40    225          17.777778
NEW ZEALAND   South Island            34    7     41          17.073171
AUSTRALIA     Victoria                72   14     86          16.279070
USA           Hawaii                 258   50    308          16.233766

La tasa de fatalidad varía considerablemente entre las diferentes localizaciones analizadas.

Entre las zonas con al menos 20 ataques y una localización identificada, South Province (New Caledonia) presenta la mayor tasa de fatalidad, con un 45%, seguida de Pernambuco (Brazil) con aproximadamente un 39,5% y Torres Strait (Australia) con un 32,9%.

También se observan diferencias importantes dentro de un mismo país. En Australia, por ejemplo, la tasa de fatalidad varía entre Torres Strait (32,9%), South Australia (25,9%), Queensland (25,4%), New South Wales (22,2%), Tasmania (18,9%), Western Australia (17,8%) y Victoria (16,3%).

Estos resultados apoyan la hipótesis de que la proporción de ataques mortales registrados varía según la localización geográfica.

In [47]:
from cleaning import ataques_por_periodo

tabla_periodos = ataques_por_periodo(df)
tabla_periodos

,Ataques,Años,Ataques_por_año
Periodo,,,
hasta 1800,49,NaN,NaN
1801-1900,585,100.0,5.8
1901-1910,155,10.0,15.5
1911-1920,149,10.0,14.9
1921-1930,218,10.0,21.8
1931-1940,273,10.0,27.3
1941-1950,302,10.0,30.2
1951-1960,517,10.0,51.7
1961-1970,567,10.0,56.7


En el análisis de ataques por año se observa que en el siglo XIX había una media de 6 ataques anuales. Estos datos deben tomarse con precaución, ya que en esa época había poca documentación y registro de los ataques.

A comienzos del siglo XX se aprecia un aumento progresivo, desde unos 16 ataques por año hasta unos 57 en la década de 1961-1970. Después se produce una caída (33 ataques por año en 1971-1980), seguida de una recuperación que alcanza un nuevo máximo de 63 ataques por año en 1991-2000.

Ya en el siglo XXI el aumento es significativo: 103 ataques por año en la primera década y 125 en la segunda. Este crecimiento se debe probablemente, en parte, a una mejor documentación de los casos.

En el último periodo (2021-2026) la media baja a 83 ataques por año. Esta bajada debe interpretarse con cautela: los casos recientes tardan en registrarse y verificarse, y el periodo aún no ha terminado, por lo que habrá que ver si la tendencia se confirma.

Conclusión: los ataques registrados por año crecen de forma clara desde el siglo XX, con un máximo en la década de 2010. Ese aumento refleja probablemente tanto un mayor número de encuentros humano-tiburón como una mejora de los registros, y el descenso reciente debe tomarse con cautela por la posible falta de datos.

In [50]:
from cleaning import clean_activity

df = clean_activity(df)
df["Activity"].value_counts().head(20)

Activity
surfing            1162
swimming           1075
unknown             588
fishing             514
spearfishing        405
wading              178
bathing             167
diving              155
snorkeling          138
standing            115
scuba diving        106
body boarding        70
boogie boarding      60
body surfing         55
kayaking             44
free diving          35
treading water       33
fell overboard       33
pearl diving         32
surf skiing          24
Name: count, dtype: int64

Dos actividades dominan los registros: el surf (1.162 casos) y la natación (1.075) suman 2.237 casos, cerca del 32% del total. Si añadimos la pesca (514) y la pesca submarina (405), estas cuatro categorías reúnen alrededor del 45%.

El surf tiene más casos probablemente porque mucha gente lo practica y pasa horas en el agua, no porque sea la actividad más arriesgada. Sin saber cuántas personas practican cada actividad, solo se puede hablar de frecuencia, no de riesgo.

La pesca representa cerca del 7% de los casos. Sin embargo, no siempre se practica dentro del agua, ya que puede hacerse desde un muelle, una barca o la orilla. Sería interesante comprobar en qué tipo de lugares ocurren estos casos.

Las actividades bajo el agua suman cerca del 12% de los registros. Destaca el spearfishing (405 casos), muy por encima del buceo con botella (106). Una posible explicación es que la captura de peces atrae a los tiburones, aunque habría que confirmarlo con más datos.

Conclusión: los ataques registrados se concentran en actividades de ocio en el agua, sobre todo surf y natación, seguidas de pesca y buceo. Probablemente el factor principal sea la exposición al mar más que la actividad concreta, por lo que hay que tener cuidado al comparar totales sin conocer cuánta gente practica cada actividad.

